# Computational Exercises 2

## Exercise 1

In [ ]:
import firedrake as fd

refinement = [16, 32, 64, 128, 512]


print(f"Num. elements \t H1norm \t\t H1norm_handmade")
for ref in refinement:
    # Finite element mesh
    Lx, Ly = 1.0, 1.0
    msh = fd.RectangleMesh(ref, ref, Lx, Ly, quadrilateral=False)

    # Space of functions
    Vd = fd.FunctionSpace(msh, "CG", degree=1)

    # Test and Trial functions
    u  = fd.TrialFunction(Vd)
    v  = fd.TestFunction(Vd)

    # Boundary conditions
    u_boundary = fd.Constant(0.0)
    bc = fd.DirichletBC(Vd, u_boundary, "on_boundary")

    # Source term
    f  = fd.Constant(1.0)

    x  = fd.SpatialCoordinate(msh)
    mu = 0.01 + fd.exp(- 100.0 * ((x[0] - 0.5)**2 + (x[1] - 0.5)**2)) # this is a function of x!


    # Bilinear form (lhs) and linear form (rhs)
    a  = fd.inner(mu * fd.grad(u), fd.grad(v)) * fd.dx
    L  = fd.inner(f, v) * fd.dx

    # Solve the problem
    ud = fd.Function(Vd)
    opts={"ksp_type": "preonly", "pc_type": "lu"}
    fd.solve(a==L, ud, bcs=[bc], solver_parameters=opts)


    L2norm = fd.sqrt(fd.assemble(fd.inner(ud, ud) * fd.dx))
    gradterms = fd.inner(fd.grad(ud), fd.grad(ud)) * fd.dx
    H1norm = fd.sqrt(L2norm + fd.assemble(gradterms))

    gradterms_hand = (ud.dx(0)**2 + ud.dx(1)**2) * fd.dx
    H1norm_hand = fd.sqrt(L2norm + fd.assemble(gradterms_hand))
    amount_elements = (ref**2)*2

    print(f"{amount_elements} \t\t {H1norm} \t {H1norm_hand}")


Num. elements 	 H1norm 		 H1norm_handmade
512 		 18.33021623072631 	 18.33021623072631
2048 		 18.454331679634286 	 18.454331679634286
8192 		 18.485328579061907 	 18.485328579061907
32768 		 18.493100083408848 	 18.493100083408848
524288 		 18.495530757302692 	 18.495530757302692


## Exercise 2

In [19]:
import firedrake as fd

msh = fd.RectangleMesh(ref, ref, Lx, Ly, quadrilateral=False)
Vd = fd.FunctionSpace(msh, "CG", degree=1)
x = fd.SpatialCoordinate(msh)
u = fd.Function(Vd)
u = fd.as_vector(
    [
        fd.sin(x[0]) * fd.sin(x[1]),
        fd.cos(x[0]) * fd.cos(x[1])
    ]
)

L2norm = fd.sqrt(fd.assemble(fd.inner(u, u) * fd.dx))
gradterms = fd.inner(fd.grad(u), fd.grad(u)) * fd.dx
H1norm = fd.sqrt(L2norm + fd.assemble(gradterms))

L2norm_hand = fd.sqrt(fd.assemble((u[0]**2 + u[1]**2) * fd.dx))
gradterms_hand = (u[0].dx(0)**2 + u[0].dx(1)**2 + u[1].dx(0)**2 + u[1].dx(1)**2) * fd.dx
H1norm_hand = fd.sqrt(L2norm_hand + fd.assemble(gradterms_hand))

print(f"H1norm \t\t H1norm_handmade")
print(f"{H1norm} \t {H1norm_hand}")

print(f"L2norm \t\t L2_handmade")
print(f"{L2norm} \t {L2norm_hand}")

tsfc:WARNING Estimated quadrature degree 12 more than tenfold greater than any argument/coefficient degree (max 1)
tsfc:WARNING Estimated quadrature degree 12 more than tenfold greater than any argument/coefficient degree (max 1)


H1norm 		 H1norm_handmade
1.2530173092529684 	 1.2530173092529684
L2norm 		 L2_handmade
0.7767578298954999 	 0.7767578298954999


## Exercise 3

$$(u,v) = \int_\Omega sin(mx)sin(nx)dx = \frac12 \int^{\pi}_{-\pi} cos((m-n)x) - cos((m+n)x) dx$$

If $m \ne n$

$$ = \left. - \frac12 \frac{sin((m-n)x)}{m-n} + \frac12 \frac{sin((m+n)x)}{m+n} \right|^\pi_{-\pi} = 0$$

If $m = n$

$$=\frac12 \int^{\pi}_{-\pi} 1 - cos((m+n)x) dx = \pi + \left. \frac{sin((m+n)x)}{m+n} \right|^\pi_{-\pi}= \pi$$

So, we have $(u,v) = \delta_{mn}$

Now numerically:

In [28]:
import firedrake as fd
import numpy as np
msh = fd.IntervalMesh(1000, -np.pi, right=np.pi)

x = fd.SpatialCoordinate(msh)

m = [1, 1, 1, 4, 1]
n = [1, 2, 3, 4, 5]
for j in range(2):
    Vd = fd.FunctionSpace(msh, "CG", j+1)
    
    u = fd.Function(Vd)
    v = fd.Function(Vd)

    for i in range(5):
        u.interpolate(fd.sin(m[i]*x[0]))
        v.interpolate(fd.sin(n[i]*x[0]))

        print(f"Inner product m={m[i]}; n={n[i]}; order of interpolation={j+1}:", fd.assemble(fd.inner(u,v)*fd.dx))
    print("")


Inner product m=1; n=1; order of interpolation=1: 3.1415719828066773
Inner product m=1; n=2; order of interpolation=1: -7.500005585859919e-17
Inner product m=1; n=3; order of interpolation=1: 3.488178556186551e-16
Inner product m=4; n=4; order of interpolation=1: 3.141261937380622
Inner product m=1; n=5; order of interpolation=1: -8.805722755920184e-17

Inner product m=1; n=1; order of interpolation=2: 3.141592653579602
Inner product m=1; n=2; order of interpolation=2: -1.1429530394515527e-16
Inner product m=1; n=3; order of interpolation=2: -1.553493299199012e-15
Inner product m=4; n=4; order of interpolation=2: 3.1415926509785024
Inner product m=1; n=5; order of interpolation=2: 3.679503446636596e-16



The results are as expected. This inner product acts as a Kronecker delta.